In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
db = pd.read_csv("Bank Customer Churn Prediction.csv")

In [3]:
db.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# data info
db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       10000 non-null  int64  
 1   credit_score      10000 non-null  int64  
 2   country           10000 non-null  object 
 3   gender            10000 non-null  object 
 4   age               10000 non-null  int64  
 5   tenure            10000 non-null  int64  
 6   balance           10000 non-null  float64
 7   products_number   10000 non-null  int64  
 8   credit_card       10000 non-null  int64  
 9   active_member     10000 non-null  int64  
 10  estimated_salary  10000 non-null  float64
 11  churn             10000 non-null  int64  
dtypes: float64(2), int64(8), object(2)
memory usage: 937.6+ KB


In [5]:
# check null values
db.isnull().sum()

customer_id         0
credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

In [6]:
# check duplicated values
db.duplicated().sum()

np.int64(0)

In [7]:
# check unique value
db["country"].unique()

array(['France', 'Spain', 'Germany'], dtype=object)

In [8]:
# check unique values
db["gender"].unique()

array(['Female', 'Male'], dtype=object)

In [9]:
# data described
db.describe()

,customer_id,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [10]:
# drop extra feature
db.drop("customer_id" , axis = 1 , inplace = True)

In [11]:
# assign x and y features
# x = independent feature
# y = dependent feature
x = db.drop("churn" , axis = 1)
y = db["churn"]

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
# split x and y into train test 
X_train , X_test , y_train , y_test = train_test_split(x , y , test_size = 0.3 , random_state = 42)

In [14]:
cat_cols = x.select_dtypes(include=['object']).columns
num_cols = x.select_dtypes(include=['int64', 'float64']).columns

In [15]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop='first', sparse_output=False , handle_unknown = 'ignore')

X_train_cat = encoder.fit_transform(X_train[cat_cols])
X_test_cat  = encoder.transform(X_test[cat_cols])

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num  = scaler.transform(X_test[num_cols])

In [17]:
X_train_final = np.hstack((X_train_cat, X_train_num))

In [18]:
X_test_final  = np.hstack((X_test_cat, X_test_num))

In [19]:
from tensorflow import keras 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import layers, activations

In [30]:
model = Sequential([
        Dense(20 , activation = 'relu' , input_dim = X_train_final.shape[1]),
        Dense(10 , activation = 'relu'),
        Dense(5  , activation = 'relu'),
        Dense(1  , activation = 'sigmoid')
])

In [31]:
model.compile(optimizer = 'adam' , loss = 'binary_crossentropy' , metrics = ['accuracy'])

In [32]:
history = model.fit(X_train_final , y_train , epochs = 100 , batch_size = 10 , validation_split = 0.20 , verbose = 1)

Epoch 1/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.7989 - loss: 0.4562 - val_accuracy: 0.8100 - val_loss: 0.4234
Epoch 2/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8180 - loss: 0.4133 - val_accuracy: 0.8221 - val_loss: 0.4036
Epoch 3/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8345 - loss: 0.3887 - val_accuracy: 0.8350 - val_loss: 0.3854
Epoch 4/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8438 - loss: 0.3717 - val_accuracy: 0.8350 - val_loss: 0.3796
Epoch 5/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8520 - loss: 0.3588 - val_accuracy: 0.8457 - val_loss: 0.3714
Epoch 6/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8577 - loss: 0.3536 - val_accuracy: 0.8471 - val_loss: 0.3668
Epoch 7/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8579 - loss: 0.3477 - val_accuracy: 0.8500 - val_loss: 0.3590
Epoch 8/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8614 - loss: 0.3423 - val_accu

In [38]:
loss , acc = model.evaluate(X_test_final , y_test , verbose = 0)

In [39]:
print(acc)
print(loss)

0.8533333539962769
0.35827505588531494


In [40]:
y_pred = model.predict(X_test_final)

print(y_test[:5])
# [1, 0, 1, 0, 1]

print(y_pred[:5])
# [0.87, 0.12, 0.65, 0.31, 0.91]

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step   
6252    0
4684    0
1731    0
4742    0
4521    0
Name: churn, dtype: int64
[[0.01903701]
 [0.00656919]
 [0.11915576]
 [0.050103  ]
 [0.08467762]]


In [42]:
from sklearn.metrics import precision_score , accuracy_score , recall_score
from sklearn.metrics import classification_report

y_pred = (model.predict(X_test_final) > 0.5).astype(int)

print(f"Accuracy        : {accuracy_score(y_test , y_pred)}")
print(f"Precision       : {precision_score(y_test , y_pred)}")
print(f"Recall          : {recall_score(y_test , y_pred)}")
print(f"Classification  : {classification_report(y_test , y_pred)}")

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Accuracy        : 0.8533333333333334
Precision       : 0.6643835616438356
Recall          : 0.4982876712328767
Classification  :               precision    recall  f1-score   support

           0       0.89      0.94      0.91      2416
           1       0.66      0.50      0.57       584

    accuracy                           0.85      3000
   macro avg       0.78      0.72      0.74      3000
weighted avg       0.84      0.85      0.85      3000

